# XGBoost Dataset-2: rest_abs_diff

This notebook starts from `XGBoost_Dataset2_Prematch.ipynb` and adds exactly one engineered pre-match feature: `rest_abs_diff`.

Feature: Absolute rest-days gap between the home and away team.

The train/test accuracy report is written to `XGBoost_Dataset2_rest_abs_diff_output`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier

In [ ]:
EXPERIMENT_FOLDER = Path('Dataset-2_rest_abs_diff')
if Path.cwd().name == EXPERIMENT_FOLDER.name:
    OUTPUT_DIR = Path.cwd()
elif EXPERIMENT_FOLDER.exists():
    OUTPUT_DIR = EXPERIMENT_FOLDER
else:
    OUTPUT_DIR = Path.cwd()

DATA_PATH = OUTPUT_DIR.parent / 'Dataset-2.csv'
if not DATA_PATH.exists():
    DATA_PATH = Path('Dataset-2.csv')

OUT_FILE = OUTPUT_DIR / 'XGBoost_Dataset2_rest_abs_diff_output'
TARGET = 'home_win'
ADDED_FEATURE = 'rest_abs_diff'
FEATURE_DESCRIPTION = 'Absolute rest-days gap between the home and away team.'

POST_MATCH_OR_LABEL_COLUMNS = [
    'home_result',
    'away_result',
    'partial_result',
    'result',
    'home_win',
    'market_favorite_won_avg',
]

NON_FEATURE_IDENTIFIER_COLUMNS = [
    'event_id',
    'encoded_event_id',
    'url',
]

RANDOM_STATE = 42
TEST_SIZE = 0.25

In [ ]:
def add_engineered_feature(df):
    df[ADDED_FEATURE] = (df['home_rest_days'] - df['away_rest_days']).abs()
    df[ADDED_FEATURE] = df[ADDED_FEATURE].replace([np.inf, -np.inf], np.nan)
    return df

In [ ]:
df = pd.read_csv(DATA_PATH).drop_duplicates().copy()

if TARGET not in df.columns:
    raise ValueError(f'Target column {TARGET!r} not found in {DATA_PATH}')

df = add_engineered_feature(df)
y = df[TARGET].astype(int)

excluded_columns = [
    c for c in POST_MATCH_OR_LABEL_COLUMNS + NON_FEATURE_IDENTIFIER_COLUMNS
    if c in df.columns
]
feature_columns = [c for c in df.columns if c not in excluded_columns]
X = df[feature_columns].copy()

print(f'Loaded {DATA_PATH.name}: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Added feature: {ADDED_FEATURE}')
print(f'Feature description: {FEATURE_DESCRIPTION}')
print(f'Pre-match feature columns used: {len(feature_columns)}')
print('Excluded columns:')
for column in excluded_columns:
    print(f'- {column}')

X[[ADDED_FEATURE]].head()

In [ ]:
numeric_features = X.select_dtypes(include=[np.number, 'bool']).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

try:
    one_hot = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown='ignore', sparse=True)

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        (
            'categorical',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', one_hot),
            ]),
            categorical_features,
        ),
    ],
    remainder='drop',
)

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

clf = Pipeline([
    ('preprocess', preprocessor),
    ('model', model),
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

clf.fit(X_train, y_train)

print(f'Numeric columns: {len(numeric_features)}')
print(f'Categorical columns: {len(categorical_features)}')
print(f'Train rows: {len(X_train)}')
print(f'Test rows: {len(X_test)}')

In [ ]:
train_pred = clf.predict(X_train)
test_pred = clf.predict(X_test)

train_accuracy = accuracy_score(y_train, train_pred)
test_accuracy = accuracy_score(y_test, test_pred)
report = classification_report(y_test, test_pred, digits=4)
cm = confusion_matrix(y_test, test_pred)

output = f'''XGBoost Dataset-2 one-feature experiment
Dataset: {DATA_PATH.name}
Added feature: {ADDED_FEATURE}
Feature description: {FEATURE_DESCRIPTION}
Rows after duplicate removal: {len(df)}
Target column: {TARGET}
Split: stratified train/test with test_size={TEST_SIZE} and random_state={RANDOM_STATE}
Train rows: {len(X_train)}
Test rows: {len(X_test)}

Train accuracy: {train_accuracy:.4f}
Test accuracy: {test_accuracy:.4f}

Input policy:
- Starts from the same pre-match feature set as XGBoost_Dataset2_Prematch.ipynb.
- Adds exactly one engineered pre-match feature: {ADDED_FEATURE}.
- Does not use post-match scores, final result text, or the target as input.

Excluded post-match/label columns:
{chr(10).join('- ' + c for c in POST_MATCH_OR_LABEL_COLUMNS if c in df.columns)}

Excluded identifier columns:
{chr(10).join('- ' + c for c in NON_FEATURE_IDENTIFIER_COLUMNS if c in df.columns)}

Feature columns used ({len(feature_columns)}):
{chr(10).join('- ' + c for c in feature_columns)}

Numeric feature columns ({len(numeric_features)}): {', '.join(numeric_features)}
Categorical feature columns ({len(categorical_features)}): {', '.join(categorical_features)}

Test classification report:
{report}

Test confusion matrix [[TN, FP], [FN, TP]]:
{cm.tolist()}
'''

OUT_FILE.write_text(output, encoding='utf-8')
print(output)
print(f'Wrote {OUT_FILE.resolve()}')